# 03 — Strategy comparison: quantitative (synthetic held-out) + qualitative (real paintings)

Compares the four hard-segmentation strategies on:

1. **Synthetic held-out** (from `dstroke_synth/val/`): IoU, F1, precision, recall with 2-px tolerance.
2. **Real authenticated paintings** (Manet, Van Gogh, Rembrandt): side-by-side visual grid.

Pre-requisite: all four 02x notebooks have run and their checkpoints exist under `outputs/dstroke/`. SAM 2 runs on-the-fly (no checkpoint to load beyond SAM's own).

In [ ]:
from __future__ import annotations
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn.functional as F

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR))
import dstroke_utils as d

REPO_ROOT = Path('C:/Users/MichelleJacobs/Repos/AIAi')
SYNTH = REPO_ROOT / 'data_notcommitted' / 'dstroke_synth'
RAW = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'data' / 'raw'
OUT = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'outputs' / 'dstroke'
CKPTS = {
    'A': OUT / 'strategy_A_pix2pix_finetune.pt',
    'B': OUT / 'strategy_B_pix2pix_scratch.pt',
    'D': OUT / 'strategy_D_segformer.pt',
}
SAM_DIR = OUT / 'strategy_C_sam2'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# reuse UNet/PatchGAN defs from 02a
nb = json.load(open(NB_DIR / '02a_pix2pix_finetune.ipynb'))
exec('\n\n'.join(''.join(c['source']) for c in nb['cells']
                  if c['cell_type']=='code' and 'history = {' not in ''.join(c['source'])
                  and 'training curves' not in ''.join(c['source'])), globals())

def load_pix2pix(ckpt_path):
    G = UNetGenerator(in_ch=3, out_ch=1).to(DEVICE)
    state = torch.load(ckpt_path, map_location=DEVICE)
    G.load_state_dict(state['G'])
    G.eval()
    return G

def load_segformer(ckpt_path):
    from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
    m = SegformerForSemanticSegmentation.from_pretrained(
        'nvidia/mit-b0', num_labels=2, ignore_mismatched_sizes=True,
    ).to(DEVICE)
    state = torch.load(ckpt_path, map_location=DEVICE)
    m.load_state_dict(state['model'])
    m.eval()
    proc = SegformerImageProcessor(do_resize=False, do_reduce_labels=False)
    return m, proc

models = {}
if CKPTS['A'].exists(): models['A'] = load_pix2pix(CKPTS['A']);          print('loaded A')
if CKPTS['B'].exists(): models['B'] = load_pix2pix(CKPTS['B']);          print('loaded B')
if CKPTS['D'].exists(): models['D'] = load_segformer(CKPTS['D']);         print('loaded D')

In [ ]:
def infer_pix2pix(G, rgb):
    x = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).to(DEVICE) * 2 - 1
    with torch.no_grad():
        g = G(x.float())
    g01 = ((g[0,0].cpu().numpy() + 1) / 2).clip(0, 1)
    t = d.trapped_ball_merge(g01).astype(np.float32)
    return g01, t

def infer_segformer(bundle, rgb):
    m, proc = bundle
    x = proc(images=(rgb*255).astype(np.uint8), return_tensors='pt')['pixel_values'].to(DEVICE)
    with torch.no_grad():
        logits = m(pixel_values=x.float()).logits
        logits = F.interpolate(logits, size=rgb.shape[:2], mode='bilinear', align_corners=False)
        probs = torch.sigmoid(logits[:, 1:2] - logits[:, 0:1])
    return probs[0,0].cpu().numpy()

def load_sam_edge(orig_name, val=True):
    sub = SAM_DIR / ('val' if val else 'real')
    candidate = list(sub.rglob(f'{Path(orig_name).stem}_edge.png'))
    if not candidate: return None
    return np.array(Image.open(candidate[0]).convert('L'), dtype=np.float32) / 255.0

## Quantitative — 50 held-out synthetic pairs

In [ ]:
val_paintings = sorted((SYNTH / 'val' / 'paintings').glob('*.png'))[:50]
rows_metric = []
for p in tqdm(val_paintings):
    rgb = np.array(Image.open(p).convert('RGB'), dtype=np.float32) / 255.0
    gt = np.array(Image.open(SYNTH / 'val' / 'edges' / p.name).convert('L'), dtype=np.float32) / 255.0
    for key in ('A', 'B'):
        if key in models:
            _, t = infer_pix2pix(models[key], rgb)
            rows_metric.append({'strategy': key, 'file': p.name, **d.compute_edge_metrics(t, gt)})
    if 'D' in models:
        pr = infer_segformer(models['D'], rgb)
        rows_metric.append({'strategy': 'D', 'file': p.name, **d.compute_edge_metrics(pr, gt)})
    sam = load_sam_edge(p.name, val=True)
    if sam is not None:
        rows_metric.append({'strategy': 'C', 'file': p.name, **d.compute_edge_metrics(sam, gt)})

df = pd.DataFrame(rows_metric)
summary = df.groupby('strategy')[['iou','f1','precision','recall']].agg(['mean','std']).round(3)
summary

## Qualitative — real authenticated paintings (3 per artist)

In [ ]:
rows_vis = []
for artist in ('manet', 'van_gogh', 'rembrandt'):
    art_dir = RAW / artist / 'authenticated'
    if not art_dir.exists(): continue
    picks = [p for ext in ('*.tif', '*.tiff', '*.png', '*.jpg') for p in art_dir.glob(ext)][:3]
    for p in picks:
        rgb_full = np.array(Image.open(p).convert('RGB').resize((256, 256)), dtype=np.float32) / 255.0
        row = {'input': rgb_full, '_label': f'{artist}/{p.name}'}
        if 'A' in models: _, row['A'] = infer_pix2pix(models['A'], rgb_full)
        if 'B' in models: _, row['B'] = infer_pix2pix(models['B'], rgb_full)
        if 'D' in models: row['D'] = infer_segformer(models['D'], rgb_full)
        sam = load_sam_edge(p.name, val=False)
        if sam is not None: row['C'] = sam
        rows_vis.append(row)

fig = d.build_comparison_grid(rows_vis, save_path=OUT / 'strategy_comparison_grid.png')
plt.show()

## Summary

Run the following cell to auto-pick the winner on synthetic F1 (override manually if the visual grid tells a different story).

In [ ]:
if len(df):
    winner = df.groupby('strategy')['f1'].mean().idxmax()
    print(f'Strategy {winner} wins on mean synthetic F1.')
else:
    winner = 'A'
    print('No metrics data yet — defaulting winner to A.')

(OUT / 'winning_strategy.txt').write_text(winner)
print(f'Wrote winner marker: {OUT / "winning_strategy.txt"} → {winner}')